In [1]:
# =========================
# 0) Imports / config
# =========================
from __future__ import annotations

import os
import math
import json
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import librosa
import scipy.fftpack as fftpack

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
SR = 16000
N_FFT = 1024
HOP_LENGTH = 160
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

TRAIN_MANIFEST = Path("manifests", "LA_train_preprocessed.csv")
DEV_MANIFEST   = Path("dev_manifest.csv")
TEST_MANIFEST  = Path("test_manifest.csv")

FEATURES_DIR = Path("features")
MODELS_DIR = Path("models")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

cpu


c:\Users\K\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================
# 1) Helpers
# =========================
def pool_mean_std(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    x: (T, D) or (D,)
    returns: (2D,) if time-varying, else (D,)
    """
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 1:
        return x
    mu = x.mean(axis=0)
    sd = x.std(axis=0)
    return np.concatenate([mu, sd], axis=0).astype(np.float32)

def ensure_2d(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return x if x.ndim == 2 else x[None, :]

def load_audio(wav_path: str | Path, sr: int = SR):
    y, _ = librosa.load(str(wav_path), sr=sr, mono=True)
    if len(y) == 0:
        raise ValueError(f"Empty audio: {wav_path}")
    return y.astype(np.float32)

def save_vec(path: Path, arr: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(str(path), arr.astype(np.float32))

def load_manifest(manifest_csv: str | Path) -> pd.DataFrame:
    df = pd.read_csv(manifest_csv)
    required = {"utt", "wav", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Manifest missing columns: {missing}")
    return df

In [3]:
# =========================
# 2) Handcrafted features
# =========================

# ---- CQCC-like extractor (close to your version) ----
def extract_cqcc_from_wav(
    wav_path,
    sr: int = SR,
    n_cq_bins: int = 84,
    hop_length: int = HOP_LENGTH,
    n_cep: int = 20,
):
    """
    CQCC-like:
      - CQT magnitude
      - log
      - DCT across frequency bins
      - keep first n_cep coefficients per frame
    Returns: (T, n_cep)
    """
    y = load_audio(wav_path, sr=sr)
    C = librosa.cqt(y, sr=sr, hop_length=hop_length, n_bins=n_cq_bins, bins_per_octave=12)
    C_mag = np.abs(C)
    C_mag[C_mag <= 1e-12] = 1e-12
    logC = np.log(C_mag)
    cep = fftpack.dct(logC, axis=0, type=2, norm="ortho")[:n_cep, :]
    return cep.T.astype(np.float32)

# ---- MFCC ----
def extract_mfcc_from_wav(
    wav_path,
    sr: int = SR,
    n_mfcc: int = 20,
    n_fft: int = N_FFT,
    hop_length: int = HOP_LENGTH,
):
    y = load_audio(wav_path, sr=sr)
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length
    )
    return mfcc.T.astype(np.float32)

# ---- LFCC ----
def _linear_filterbank(sr: int, n_fft: int, n_filters: int, fmin: float = 0.0, fmax: float | None = None):
    if fmax is None:
        fmax = sr / 2
    # linear spacing in Hz
    hz_points = np.linspace(fmin, fmax, n_filters + 2)
    bins = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    fb = np.zeros((n_filters, n_fft // 2 + 1), dtype=np.float32)

    for m in range(1, n_filters + 1):
        left, center, right = bins[m - 1], bins[m], bins[m + 1]
        left = max(left, 0)
        right = min(right, n_fft // 2)
        center = np.clip(center, left + 1, right - 1) if right > left + 2 else center

        for k in range(left, center):
            fb[m - 1, k] = (k - left) / max(center - left, 1)
        for k in range(center, right):
            fb[m - 1, k] = (right - k) / max(right - center, 1)

    return fb

def extract_lfcc_from_wav(
    wav_path,
    sr: int = SR,
    n_lfcc: int = 20,
    n_filters: int = 40,
    n_fft: int = N_FFT,
    hop_length: int = HOP_LENGTH,
):
    y = load_audio(wav_path, sr=sr)
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) ** 2
    fb = _linear_filterbank(sr=sr, n_fft=n_fft, n_filters=n_filters)
    E = np.dot(fb, S)
    E[E <= 1e-12] = 1e-12
    logE = np.log(E)
    lfcc = fftpack.dct(logE, axis=0, type=2, norm="ortho")[:n_lfcc, :]
    return lfcc.T.astype(np.float32)

# ---- PLP (wrapper; adjust one call if your installed spafe version differs) ----
def extract_plp_from_wav(
    wav_path,
    sr: int = SR,
    n_plp: int = 20,
):
    """
    PLP via spafe.
    pip install spafe
    """
    # try:
    #     from spafe.features.rplp import plp as spafe_plp
    # except Exception as e:
    #     raise ImportError("PLP requires `spafe`. Install it with: pip install spafe") from e

    # y = load_audio(wav_path, sr=sr)

    # # Different spafe versions expose slightly different parameter names.
    # # This tries a few common variants so you only need a tiny edit if needed.
    # candidates = [
    #     dict(fs=sr, num_ceps=n_plp),
    #     dict(fs=sr, order=n_plp),
    #     dict(fs=sr, num_ceps=n_plp, order=n_plp),
    # ]

    # feats = None
    # last_err = None
    # for kwargs in candidates:
    #     try:
    #         feats = spafe_plp(y, **kwargs)
    #         break
    #     except TypeError as e:
    #         last_err = e

    # if feats is None:
    #     raise RuntimeError(
    #         "Could not call spafe.features.plp.plp with the current version. "
    #         "Edit only extract_plp_from_wav() to match your installed spafe API."
    #     ) from last_err

    # feats = np.asarray(feats, dtype=np.float32)
    # return ensure_2d(feats)
    try:
        from spafe.features.rplp import plp as spafe_plp
    except Exception as e:
        raise ImportError(
            "PLP requires `spafe`. Install it with: pip install spafe"
        ) from e

    y = load_audio(wav_path, sr=sr)

    feats = spafe_plp(
        y,
        fs=sr,
        num_ceps=n_plp,
        modelorder=n_plp,
    )

    feats = np.asarray(feats, dtype=np.float32)
    return ensure_2d(feats)

HANDCRAFTED_EXTRACTORS = {
    "cqcc": extract_cqcc_from_wav,
    "lfcc": extract_lfcc_from_wav,
    "mfcc": extract_mfcc_from_wav,
    "plp": extract_plp_from_wav,
}

def extract_and_save_features(
    manifest_csv: str | Path,
    out_dir: str | Path,
    extractor_fn,
    pooled: bool = True,
    save_frames: bool = False,
    suffix: str = ".npy",
    feature_manifest = None
):
    """
    Reads manifest with columns: utt,wav,label
    Saves one .npy per utterance.
    Returns a new feature manifest:
        utt, feat_path, label
    """
    if feature_manifest:
        return feature_manifest
    manifest_csv = Path(manifest_csv)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = load_manifest(manifest_csv)
    rows = []

    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Extracting {out_dir.name}"):
        utt = str(r["utt"])
        wav = r["wav"]
        label = int(r["label"])

        try:
            frames = extractor_fn(wav)
            frames = ensure_2d(frames)

            if save_frames:
                save_vec(out_dir / f"{utt}_frames.npy", frames)

            vec = pool_mean_std(frames) if pooled else frames
            feat_path = out_dir / f"{utt}{suffix}"
            save_vec(feat_path, vec)

            rows.append((utt, str(feat_path.resolve()), label))
        except Exception as e:
            print(f"[FAIL] {utt}: {e}")

    out_manifest = out_dir.parent / f"{out_dir.name}_{manifest_csv.stem}_feat_manifest.csv"
    pd.DataFrame(rows, columns=["utt", "feat_path", "label"]).to_csv(out_manifest, index=False)
    print(f"Saved feature manifest -> {out_manifest}")
    return out_manifest

In [4]:
# =========================
# 3) Deep embeddings (Wav2Vec2 / HuBERT)
# =========================
def load_ssl_model(model_name: str, device: str = DEVICE):
    """
    model_name examples:
      - facebook/wav2vec2-base
      - facebook/hubert-base-ls960
    """
    try:
        from transformers import AutoProcessor, AutoModel
    except Exception as e:
        raise ImportError("Install transformers: pip install transformers") from e

    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    return processor, model

@torch.no_grad()
def extract_ssl_embedding_from_wav(
    wav_path,
    processor,
    model,
    sr: int = SR,
    device: str = DEVICE,
    pooling: str = "meanstd",   # "mean" or "meanstd"
):
    y = load_audio(wav_path, sr=sr)

    inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = model(**inputs)
    hidden = out.last_hidden_state.squeeze(0).detach().cpu().numpy()  # (T, D)

    if pooling == "mean":
        return hidden.mean(axis=0).astype(np.float32)
    elif pooling == "meanstd":
        return pool_mean_std(hidden)
    else:
        raise ValueError("pooling must be 'mean' or 'meanstd'")

def extract_and_save_ssl_embeddings(
    manifest_csv: str | Path,
    out_dir: str | Path,
    model_name: str,
    pooling: str = "meanstd",
):
    processor, model = load_ssl_model(model_name, device=DEVICE)

    fn = partial(
        extract_ssl_embedding_from_wav,
        processor=processor,
        model=model,
        sr=SR,
        device=DEVICE,
        pooling=pooling,
    )
    return extract_and_save_features(manifest_csv, out_dir, fn, pooled=True, save_frames=False)

In [5]:
# =========================
# 4) Shared classifier
# =========================
class FeatureNPYDataset(Dataset):
    def __init__(self, manifest_csv: str | Path):
        self.df = pd.read_csv(manifest_csv)
        if not {"feat_path", "label"}.issubset(self.df.columns):
            raise ValueError("Feature manifest must contain feat_path,label")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        x = np.load(r["feat_path"]).astype(np.float32)
        y = np.float32(r["label"])
        return torch.from_numpy(x), torch.tensor(y)

class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def infer_input_dim(manifest_csv: str | Path) -> int:
    df = pd.read_csv(manifest_csv)
    x = np.load(df.iloc[0]["feat_path"]).astype(np.float32)
    return int(x.shape[0])

def get_pos_weight(manifest_csv: str | Path) -> torch.Tensor:
    df = pd.read_csv(manifest_csv)
    y = df["label"].astype(int).values
    n_pos = max(y.sum(), 1)
    n_neg = max((1 - y).sum(), 1)
    return torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)

def eer_score(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return float(eer)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_prob = [], []

    for x, y in loader:
        x = x.to(DEVICE).float()
        logits = model(x)
        prob = torch.sigmoid(logits).detach().cpu().numpy()

        y_true.extend(y.numpy().tolist())
        y_prob.extend(prob.tolist())

    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        "eer": eer_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan"),
        "accuracy": accuracy_score(y_true, y_pred),
    }
    return metrics, y_true, y_prob

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(DEVICE).float()
        y = y.to(DEVICE).float()

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

def make_loaders(train_manifest, dev_manifest = None, test_manifest = None, batch_size: int = 64):
    train_ds = FeatureNPYDataset(train_manifest)
    if dev_manifest:
        dev_ds   = FeatureNPYDataset(dev_manifest)
    if test_manifest:
        test_ds  = FeatureNPYDataset(test_manifest)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    dev_loader, test_loader = None, None
    if dev_manifest:
        dev_loader = DataLoader(dev_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    if test_manifest:
        test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, dev_loader, test_loader

def train_classifier(
    exp_name: str,
    train_manifest: str | Path,
    dev_manifest: str | Path = None,
    test_manifest: str | Path = None,
    batch_size: int = 64,
    hidden_dim: int = 256,
    dropout: float = 0.3,
    lr: float = 1e-3,
    epochs: int = 20,
):
    input_dim = infer_input_dim(train_manifest)
    model = MLPClassifier(input_dim=input_dim, hidden_dim=hidden_dim, dropout=dropout).to(DEVICE)

    train_loader, dev_loader, test_loader = make_loaders(
        train_manifest, dev_manifest, test_manifest, batch_size=batch_size
    )

    pos_weight = get_pos_weight(train_manifest)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_dev_eer = 1e9
    best_path = MODELS_DIR / f"{exp_name}_best.pt"
    final_path = MODELS_DIR / f"{exp_name}_final.pt"
    print(f"Starting training for {exp_name}, best model: {best_path}, final model: {final_path}")
    for epoch in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)

        if dev_loader:
            dev_metrics, _, _ = evaluate(model, dev_loader)

            if dev_metrics["eer"] < best_dev_eer:
                best_dev_eer = dev_metrics["eer"]
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "input_dim": input_dim,
                        "hidden_dim": hidden_dim,
                        "dropout": dropout,
                        "exp_name": exp_name,
                    },
                    best_path,
                )

        print(
            f"[{exp_name}] epoch {epoch:02d} | "
            f"train_loss={tr_loss:.4f} | dev_EER={dev_metrics['eer']:.4f} | "
            f"dev_AUC={dev_metrics['roc_auc']:.4f} | dev_ACC={dev_metrics['accuracy']:.4f}"
        )

    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    
    if test_loader:
        test_metrics, y_true, y_prob = evaluate(model, test_loader)

        print(f"\n[{exp_name}] TEST")
        print(json.dumps(test_metrics, indent=2))

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "input_dim": input_dim,
            "hidden_dim": hidden_dim,
            "dropout": dropout,
            "exp_name": f"{exp_name}_final",
        },
        final_path,
    )

    return model, test_metrics, best_path

In [ ]:
model, metrics, ckpt = train_classifier("cqcc", "scratch1/kodachi/project/manifests/cqcc_LA_train_preprocessed_feat_manifest.csv", epochs=5)

Starting training for cqcc, best model: models\cqcc_best.pt, final model: models\cqcc_final.pt


In [ ]:

# =========================
# 5) Example usage
# =========================

# ---- A) Handcrafted features ----
# Choose one extractor at a time and keep the rest of the training code unchanged.
#
# cqcc_train = extract_and_save_features(TRAIN_MANIFEST, FEATURES_DIR / "cqcc", HANDCRAFTED_EXTRACTORS["cqcc"],feature_manifest="features/cqcc_LA_train_preprocessed_feat_manifest.csv")
# cqcc_dev   = extract_and_save_features(DEV_MANIFEST,   FEATURES_DIR / "cqcc", HANDCRAFTED_EXTRACTORS["cqcc"])
# cqcc_test  = extract_and_save_features(TEST_MANIFEST,  FEATURES_DIR / "cqcc", HANDCRAFTED_EXTRACTORS["cqcc"])
# model, metrics, ckpt = train_classifier(cqcc_train, cqcc_dev, cqcc_test, exp_name="cqcc")

# mfcc_train = extract_and_save_features(TRAIN_MANIFEST, FEATURES_DIR / "mfcc", HANDCRAFTED_EXTRACTORS["mfcc"])
# mfcc_dev   = extract_and_save_features(DEV_MANIFEST,   FEATURES_DIR / "mfcc", HANDCRAFTED_EXTRACTORS["mfcc"])
# mfcc_test  = extract_and_save_features(TEST_MANIFEST,  FEATURES_DIR / "mfcc", HANDCRAFTED_EXTRACTORS["mfcc"])
# model, metrics, ckpt = train_classifier(mfcc_train, mfcc_dev, mfcc_test, exp_name="mfcc")

# lfcc_train = extract_and_save_features(TRAIN_MANIFEST, FEATURES_DIR / "lfcc", HANDCRAFTED_EXTRACTORS["lfcc"])
# lfcc_dev   = extract_and_save_features(DEV_MANIFEST,   FEATURES_DIR / "lfcc", HANDCRAFTED_EXTRACTORS["lfcc"])
# lfcc_test  = extract_and_save_features(TEST_MANIFEST,  FEATURES_DIR / "lfcc", HANDCRAFTED_EXTRACTORS["lfcc"])
# model, metrics, ckpt = train_classifier(lfcc_train, lfcc_dev, lfcc_test, exp_name="lfcc")

# plp_train = extract_and_save_features(TRAIN_MANIFEST, FEATURES_DIR / "plp", HANDCRAFTED_EXTRACTORS["plp"])
# plp_dev   = extract_and_save_features(DEV_MANIFEST,   FEATURES_DIR / "plp", HANDCRAFTED_EXTRACTORS["plp"])
# plp_test  = extract_and_save_features(TEST_MANIFEST,  FEATURES_DIR / "plp", HANDCRAFTED_EXTRACTORS["plp"])
# model, metrics, ckpt = train_classifier(plp_train, plp_dev, plp_test, exp_name="plp")


# ---- B) Deep embeddings ----
# Wav2Vec2
# wav2vec_train = extract_and_save_ssl_embeddings(TRAIN_MANIFEST, FEATURES_DIR / "wav2vec2", "facebook/wav2vec2-base", pooling="meanstd")
# wav2vec_dev   = extract_and_save_ssl_embeddings(DEV_MANIFEST,   FEATURES_DIR / "wav2vec2", "facebook/wav2vec2-base", pooling="meanstd")
# wav2vec_test  = extract_and_save_ssl_embeddings(TEST_MANIFEST,  FEATURES_DIR / "wav2vec2", "facebook/wav2vec2-base", pooling="meanstd")
# model, metrics, ckpt = train_classifier(wav2vec_train, wav2vec_dev, wav2vec_test, exp_name="wav2vec2_base")

# HuBERT
# hubert_train = extract_and_save_ssl_embeddings(TRAIN_MANIFEST, FEATURES_DIR / "hubert", "facebook/hubert-base-ls960", pooling="meanstd")
# hubert_dev   = extract_and_save_ssl_embeddings(DEV_MANIFEST,   FEATURES_DIR / "hubert", "facebook/hubert-base-ls960", pooling="meanstd")
# hubert_test  = extract_and_save_ssl_embeddings(TEST_MANIFEST,  FEATURES_DIR / "hubert", "facebook/hubert-base-ls960", pooling="meanstd")
# model, metrics, ckpt = train_classifier(hubert_train, hubert_dev, hubert_test, exp_name="hubert_base")